# Radar de tendencias — Salud / Laboratorios de investigación (v3)
**Etapas 1 y 2.** Menú *Entorno de ejecución → Ejecutar todo*. Cada dato queda con **fuente, URL y fecha**.

- **Etapa 1:** recolecta papers, preprints, ensayos, regulación y noticias (incluye una pasada con instituciones argentinas).
- **Etapa 2:** filtra lo relevante, Gemini extrae tendencias **solo con el texto de cada ítem y una cita textual**, y un verificador descarta lo que no se pueda comprobar.

Para la etapa 2 hace falta el secreto `GEMINI_API_KEY` (🔑 → Agregar nuevo secreto → pegar la clave de Google AI Studio → activar *Acceso desde el cuaderno*).

**Novedad v3:** el cuaderno prueba qué modelo de Gemini responde con tu clave (evita los modelos *preview*, que no están en el plan gratuito).

## 0. Preparar archivos

In [ ]:
!mkdir -p /content/radar/agent /content/radar/data/raw /content/radar/data/processed

In [ ]:
%%writefile /content/radar/agent/sources.json
{
  "_nota": "Cada fuente tiene id, tipo y justificacion. enabled=false o verify=true: la URL no fue confirmada, el recolector la prueba y reporta.",
  "topics": [
    "CRISPR gene editing",
    "single-cell spatial transcriptomics",
    "AI protein structure drug discovery",
    "laboratory automation robotics",
    "digital twin biology",
    "bioinformatics cloud computing genomics",
    "mRNA therapeutics platform",
    "synthetic biology biofoundry",
    "health data interoperability research",
    "machine learning clinical research"
  ],
  "api_sources": [
    {
      "id": "pubmed",
      "type": "paper",
      "why": "Base de papers biomedicos revisados; senal de madurez cientifica. Incluye una pasada con afiliacion Argentina"
    },
    {
      "id": "medrxiv",
      "type": "preprint",
      "why": "Senal temprana (preprints sin revision de pares: se marca como tal)"
    },
    {
      "id": "arxiv",
      "type": "preprint",
      "why": "IA y computacion aplicada a biologia"
    },
    {
      "id": "openalex",
      "type": "papers",
      "why": "Cobertura abierta de publicaciones y afiliaciones; permite medir crecimiento por tema"
    },
    {
      "id": "clinicaltrials",
      "type": "ensayos",
      "why": "Madurez clinica real de una tecnologia (fase, estado)"
    }
  ],
  "rss_sources": [
    {
      "id": "stat",
      "type": "noticia",
      "region": "global",
      "url": "https://www.statnews.com/feed/",
      "why": "Periodismo de salud y biotecnologia de referencia"
    },
    {
      "id": "fierce_biotech",
      "type": "noticia",
      "region": "global",
      "url": "https://www.fiercebiotech.com/rss/xml",
      "why": "Probada 13/11/2026: HTTP 403 (bloquea pedidos automaticos). Desactivada; limitacion a declarar.",
      "enabled": false
    },
    {
      "id": "nature_news",
      "type": "noticia",
      "region": "global",
      "url": "https://www.nature.com/nature.rss",
      "why": "Probada 13/11/2026: respondio 0 items. Desactivada hasta confirmar URL de feed.",
      "enabled": false
    },
    {
      "id": "nature_biotech",
      "type": "paper",
      "region": "global",
      "url": "https://www.nature.com/nbt.rss",
      "why": "Probada 13/11/2026: la respuesta no es un feed valido (ParseError). Desactivada.",
      "enabled": false
    },
    {
      "id": "fda_press",
      "type": "regulacion",
      "region": "EE.UU.",
      "url": "https://www.fda.gov/about-fda/contact-fda/stay-informed/rss-feeds/press-releases/rss.xml",
      "verify": true,
      "why": "Regulacion FDA"
    },
    {
      "id": "ema_news",
      "type": "regulacion",
      "region": "UE",
      "url": "https://www.ema.europa.eu/en/news.xml",
      "verify": true,
      "why": "Regulacion EMA"
    },
    {
      "id": "who_news",
      "type": "regulacion",
      "region": "global",
      "url": "https://www.who.int/rss-feeds/news-english.xml",
      "verify": true,
      "why": "Politica sanitaria OMS"
    },
    {
      "id": "anmat",
      "type": "regulacion",
      "region": "Argentina",
      "url": "",
      "enabled": false,
      "verify": true,
      "why": "Sin feed RSS oficial hallado. Se cubre Argentina via PubMed (afiliacion) y OpenAlex (pais AR). Pendiente: cargar noticias de argentina.gob.ar/anmat y conicet.gov.ar a mano o con WebFetch"
    },
    {
      "id": "conicet",
      "type": "noticia",
      "region": "Argentina",
      "url": "",
      "enabled": false,
      "verify": true,
      "why": "Sin feed RSS oficial hallado. Se cubre Argentina via PubMed (afiliacion) y OpenAlex (pais AR). Pendiente: cargar noticias de argentina.gob.ar/anmat y conicet.gov.ar a mano o con WebFetch"
    }
  ]
}

In [ ]:
%%writefile /content/radar/agent/collect.py
#!/usr/bin/env python3
"""Etapa 1 - Recolector. Solo biblioteca estandar (no hay que instalar nada).
Uso:  python collect.py            -> corre todas las fuentes
      python collect.py --days 30  -> ventana de dias
      python collect.py --selftest -> prueba los parsers con ejemplos locales
Guarda en ../data/raw/raw_items.jsonl (un item por linea, sin duplicados) y
../data/raw/run_log.jsonl (que fuente respondio y cuantos items trajo)."""
import argparse, hashlib, json, re, sys, time, urllib.parse, urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
RAW = ROOT / "data" / "raw"
UA = {"User-Agent": "radar-salud-tp/1.0 (trabajo practico universitario)"}
NOW = lambda: datetime.now(timezone.utc).isoformat(timespec="seconds")

def fetch(url, timeout=25):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read().decode("utf-8", "replace")

def item(source, stype, url, title, text, published, query="", region="global"):
    return {"id": hashlib.sha1(url.encode()).hexdigest()[:16], "source": source, "source_type": stype,
            "url": url, "title": (title or "").strip(), "text": re.sub(r"\s+", " ", text or "").strip()[:4000],
            "published": published or "", "retrieved_at": NOW(), "query": query, "region": region}

def strip_tags(s): return re.sub(r"<[^>]+>", " ", s or "")

# ---------- parsers (reciben texto de la respuesta, devuelven items) ----------
def parse_pubmed(esearch_json, efetch_xml, q, region="global"):
    out = []
    root = ET.fromstring(efetch_xml)
    for art in root.iter("PubmedArticle"):
        pmid = art.findtext(".//PMID")
        title = "".join(art.find(".//ArticleTitle").itertext()) if art.find(".//ArticleTitle") is not None else ""
        abst = " ".join("".join(a.itertext()) for a in art.findall(".//AbstractText"))
        y = art.findtext(".//PubDate/Year") or art.findtext(".//PubDate/MedlineDate") or ""
        if pmid and title:
            out.append(item("pubmed", "paper", f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/", title, abst, y, q, region))
    return out

def parse_clinicaltrials(js, q):
    out = []
    for s in json.loads(js).get("studies", []):
        p = s.get("protocolSection", {})
        nct = p.get("identificationModule", {}).get("nctId")
        if not nct: continue
        t = p.get("identificationModule", {}).get("briefTitle", "")
        summ = p.get("descriptionModule", {}).get("briefSummary", "")
        phase = ",".join(p.get("designModule", {}).get("phases", []) or [])
        status = p.get("statusModule", {}).get("overallStatus", "")
        d = p.get("statusModule", {}).get("lastUpdatePostDateStruct", {}).get("date", "")
        out.append(item("clinicaltrials", "ensayo", f"https://clinicaltrials.gov/study/{nct}", t,
                        f"[Fase: {phase or 's/d'}; Estado: {status}] {summ}", d, q))
    return out

def abstract_from_inverted(inv):
    if not inv: return ""
    pos = {}
    for w, idxs in inv.items():
        for i in idxs: pos[i] = w
    return " ".join(pos[i] for i in sorted(pos))

def parse_openalex(js, q, region="global"):
    out = []
    for w in json.loads(js).get("results", []):
        url = w.get("doi") or w.get("id")
        if not url or not w.get("title"): continue
        out.append(item("openalex", "paper", url, w["title"], abstract_from_inverted(w.get("abstract_inverted_index")),
                        w.get("publication_date", ""), q, region))
    return out

def parse_medrxiv(js, topics):
    out = []
    low = [t.lower() for t in topics]
    kw = {w for t in low for w in t.split() if len(w) > 4}
    for r in json.loads(js).get("collection", []):
        blob = (r.get("title", "") + " " + r.get("abstract", "")).lower()
        if any(k in blob for k in kw):
            out.append(item("medrxiv", "preprint", "https://doi.org/" + r["doi"], r.get("title"), r.get("abstract"), r.get("date"), "filtro por palabras"))
    return out

def parse_arxiv(xml, q):
    ns = {"a": "http://www.w3.org/2005/Atom"}
    out = []
    for e in ET.fromstring(xml).findall("a:entry", ns):
        url = e.findtext("a:id", namespaces=ns)
        out.append(item("arxiv", "preprint", url, e.findtext("a:title", namespaces=ns), e.findtext("a:summary", namespaces=ns), (e.findtext("a:published", namespaces=ns) or "")[:10], q))
    return out

def parse_rss(xml, src):
    out = []
    root = ET.fromstring(xml)
    for it in list(root.iter("item")) + list(root.iter("{http://www.w3.org/2005/Atom}entry")):
        link = it.findtext("link") or ""
        if not link:
            l = it.find("{http://www.w3.org/2005/Atom}link")
            link = l.get("href") if l is not None else ""
        title = it.findtext("title") or it.findtext("{http://www.w3.org/2005/Atom}title") or ""
        desc = strip_tags(it.findtext("description") or it.findtext("{http://www.w3.org/2005/Atom}summary") or "")
        pub = it.findtext("pubDate") or it.findtext("{http://www.w3.org/2005/Atom}updated") or ""
        if link: out.append(item(src["id"], src["type"], link.strip(), title, desc, pub, "rss", src.get("region", "global")))
    return out

# ---------- recolectores ----------
def run_pubmed(topics, days):
    out = []
    for q0 in topics + ["AR::" + x for x in topics]:
        ar = q0.startswith("AR::"); q = q0[4:] if ar else q0
        term = f"({q}) AND Argentina[Affiliation]" if ar else q
        base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
        ids = json.loads(fetch(base + "esearch.fcgi?" + urllib.parse.urlencode({"db": "pubmed", "term": term, "retmax": 8 if ar else 15, "sort": "date", "datetype": "pdat", "reldate": days, "retmode": "json"})))["esearchresult"]["idlist"]
        time.sleep(0.4)
        if ids:
            xml = fetch(base + "efetch.fcgi?" + urllib.parse.urlencode({"db": "pubmed", "id": ",".join(ids), "retmode": "xml"}))
            out += parse_pubmed(None, xml, q, "Argentina" if ar else "global")
            time.sleep(0.4)
    return out

def run_clinicaltrials(topics, days):
    out = []
    for q in topics:
        u = "https://clinicaltrials.gov/api/v2/studies?" + urllib.parse.urlencode({"query.term": q, "pageSize": 15, "sort": "LastUpdatePostDate:desc"})
        out += parse_clinicaltrials(fetch(u), q)
    return out

def run_openalex(topics, days):
    out = []
    since = (datetime.now() - timedelta(days=days)).strftime("%Y-%m-%d")
    for q in topics:
        for ar in (False, True):
            flt = f"from_publication_date:{since}" + (",institutions.country_code:AR" if ar else "")
            u = "https://api.openalex.org/works?" + urllib.parse.urlencode({"search": q, "filter": flt, "per-page": 8 if ar else 15, "sort": "cited_by_count:desc"})
            out += parse_openalex(fetch(u), q, "Argentina" if ar else "global")
    return out

def run_medrxiv(topics, days):
    d2 = datetime.now(); d1 = d2 - timedelta(days=min(days, 14))
    js = fetch(f"https://api.medrxiv.org/details/medrxiv/{d1:%Y-%m-%d}/{d2:%Y-%m-%d}/0")
    return parse_medrxiv(js, topics)

def run_arxiv(topics, days):
    out = []
    for q in topics:
        u = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode({"search_query": "all:" + q, "max_results": 10, "sortBy": "submittedDate", "sortOrder": "descending"})
        out += parse_arxiv(fetch(u), q); time.sleep(3)
    return out

API = {"pubmed": run_pubmed, "clinicaltrials": run_clinicaltrials, "openalex": run_openalex, "medrxiv": run_medrxiv, "arxiv": run_arxiv}

def save(items):
    RAW.mkdir(parents=True, exist_ok=True)
    f = RAW / "raw_items.jsonl"
    have = set()
    if f.exists():
        have = {json.loads(l)["id"] for l in f.read_text(encoding="utf-8").splitlines() if l.strip()}
    new = [i for i in items if i["id"] not in have]
    with f.open("a", encoding="utf-8") as fh:
        for i in new: fh.write(json.dumps(i, ensure_ascii=False) + "\n")
    return len(new)

def main(days):
    cfg = json.loads((Path(__file__).parent / "sources.json").read_text(encoding="utf-8"))
    log, allitems = [], []
    jobs = [(s["id"], lambda s=s: API[s["id"]](cfg["topics"], days)) for s in cfg["api_sources"]]
    for s in cfg["rss_sources"]:
        if s.get("enabled") is False: log.append({"source": s["id"], "ok": False, "n": 0, "note": "deshabilitada (URL pendiente)"}); continue
        jobs.append((s["id"], lambda s=s: parse_rss(fetch(s["url"]), s)))
    for sid, job in jobs:
        try:
            its = job(); allitems += its
            log.append({"source": sid, "ok": True, "n": len(its)})
        except Exception as e:
            log.append({"source": sid, "ok": False, "n": 0, "note": f"{type(e).__name__}: {e}"})
    new = save(allitems)
    with (RAW / "run_log.jsonl").open("a", encoding="utf-8") as fh:
        fh.write(json.dumps({"run_at": NOW(), "days": days, "nuevos": new, "fuentes": log}, ensure_ascii=False) + "\n")
    print(f"{'FUENTE':16}{'ESTADO':8}ITEMS")
    for l in log: print(f"{l['source']:16}{'OK' if l['ok'] else 'FALLA':8}{l['n']}  {l.get('note','')}")
    print(f"\nItems nuevos guardados: {new}  ->  {RAW/'raw_items.jsonl'}")

def selftest():
    pm = '<PubmedArticleSet><PubmedArticle><MedlineCitation><PMID>1</PMID><Article><ArticleTitle>T pubmed</ArticleTitle><Abstract><AbstractText>abs uno</AbstractText></Abstract><Journal><JournalIssue><PubDate><Year>2026</Year></PubDate></JournalIssue></Journal></Article></MedlineCitation></PubmedArticle></PubmedArticleSet>'
    ct = json.dumps({"studies": [{"protocolSection": {"identificationModule": {"nctId": "NCT01", "briefTitle": "Ensayo X"}, "descriptionModule": {"briefSummary": "resumen"}, "designModule": {"phases": ["PHASE2"]}, "statusModule": {"overallStatus": "RECRUITING", "lastUpdatePostDateStruct": {"date": "2026-10-01"}}}}]})
    oa = json.dumps({"results": [{"id": "https://openalex.org/W1", "doi": "https://doi.org/10.1/x", "title": "Paper OA", "publication_date": "2026-10-02", "abstract_inverted_index": {"hola": [0], "mundo": [1]}}]})
    mr = json.dumps({"collection": [{"title": "CRISPR editing study", "abstract": "gene editing", "doi": "10.1101/abc", "date": "2026-10-03"}, {"title": "Otro tema", "abstract": "nada", "doi": "10.1101/zzz", "date": "2026-10-03"}]})
    ax = '<feed xmlns="http://www.w3.org/2005/Atom"><entry><id>http://arxiv.org/abs/1</id><title>AX t</title><summary>s</summary><published>2026-10-04T00:00:00Z</published></entry></feed>'
    rs = '<rss><channel><item><title>Noticia</title><link>https://ex.com/n1</link><description>&lt;p&gt;Hola&lt;/p&gt;</description><pubDate>Mon, 02 Nov 2026</pubDate></item></channel></rss>'
    tests = [("pubmed", parse_pubmed(None, pm, "q"), 1), ("clinicaltrials", parse_clinicaltrials(ct, "q"), 1), ("openalex", parse_openalex(oa, "q"), 1),
             ("medrxiv(filtra)", parse_medrxiv(mr, ["CRISPR gene editing"]), 1), ("arxiv", parse_arxiv(ax, "q"), 1), ("rss", parse_rss(rs, {"id": "x", "type": "noticia"}), 1)]
    ok = True
    for n, r, exp in tests:
        good = len(r) == exp and all(i["url"].startswith("http") and i["title"] for i in r)
        ok &= good; print(("OK   " if good else "FALLA"), n, len(r))
    assert parse_openalex(oa, "q")[0]["text"] == "hola mundo"
    print("\nSELFTEST", "OK" if ok else "FALLO"); sys.exit(0 if ok else 1)

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--days", type=int, default=30); ap.add_argument("--selftest", action="store_true")
    a = ap.parse_args()
    selftest() if a.selftest else main(a.days)

In [ ]:
%%writefile /content/radar/agent/taxonomy.json
{
  "horizons": ["corto", "mediano", "largo"],
  "horizon_definition": {"corto": "0-2 anios", "mediano": "2-5 anios", "largo": "5-10+ anios"},
  "hype_phases": ["Innovation Trigger", "Peak of Inflated Expectations", "Trough of Disillusionment", "Slope of Enlightenment", "Plateau of Productivity"],
  "categories": {
    "ia_descubrimiento": {"label": "IA y modelos para descubrimiento", "keywords": ["machine learning", "deep learning", "foundation model", "large language model", "alphafold", "protein design", "drug discovery", "generative ai", "neural network", "artificial intelligence"]},
    "edicion_genica_terapias": {"label": "Edicion genica y terapias avanzadas", "keywords": ["crispr", "gene editing", "base editing", "prime editing", "gene therapy", "mrna", "car-t", "cell therapy", "aav", "oligonucleotide"]},
    "omicas": {"label": "Omicas, celula unica y espacial", "keywords": ["single-cell", "single cell", "spatial transcriptomics", "spatial omics", "multi-omics", "proteomics", "metabolomics", "sequencing", "genomics", "long-read"]},
    "automatizacion_lab": {"label": "Automatizacion y laboratorios autonomos", "keywords": ["laboratory automation", "lab automation", "robotic", "self-driving lab", "autonomous lab", "high-throughput", "lims", "liquid handling", "microfluidic"]},
    "datos_computo": {"label": "Bioinformatica, datos y computo", "keywords": ["bioinformatics", "cloud computing", "high-performance computing", "fair data", "data sharing", "interoperability", "open data", "workflow", "quantum computing"]},
    "biologia_sintetica": {"label": "Biologia sintetica y biomanufactura", "keywords": ["synthetic biology", "biofoundry", "biomanufacturing", "cell-free", "metabolic engineering", "biosensor"]},
    "gemelos_simulacion": {"label": "Gemelos digitales y simulacion", "keywords": ["digital twin", "in silico", "computational model", "simulation", "organ-on-chip", "organoid"]},
    "regulacion_seguridad": {"label": "Regulacion, etica y seguridad de datos", "keywords": ["regulation", "regulatory", "guidance", "approval", "cybersecurity", "privacy", "data protection", "ethics", "biosecurity", "policy"]},
    "ensayos_precision": {"label": "Ensayos clinicos y medicina de precision", "keywords": ["clinical trial", "precision medicine", "biomarker", "personalized", "companion diagnostic", "decentralized trial", "real-world data"]}
  },
  "evidence_level": {"paper": "revisado por pares", "ensayo": "registro oficial", "regulacion": "fuente oficial", "preprint": "preprint (sin revision de pares)", "noticia": "prensa"}
}

In [ ]:
%%writefile /content/radar/agent/process.py
#!/usr/bin/env python3
"""Etapa 2 - Filtro, extraccion con Gemini y VERIFICACION.
  python process.py filter [--max 250]
  python process.py extract [--limit 40] [--model auto]
  python process.py verify
  python process.py selftest
Regla: Gemini solo puede usar el texto del item. Todo lo que no se pueda comprobar contra el texto crudo se descarta o se marca, y queda en verification_log.jsonl."""
import argparse, json, os, re, sys, time, unicodedata, urllib.request, urllib.error
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
RAWF = ROOT / "data" / "raw" / "raw_items.jsonl"
PROC = ROOT / "data" / "processed"
TAX = json.loads((Path(__file__).parent / "taxonomy.json").read_text(encoding="utf-8"))
NOW = lambda: datetime.now(timezone.utc).isoformat(timespec="seconds")

def read_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []
def write_jsonl(p, rows, mode="w"):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    with open(p, mode, encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
def norm(s):
    s = unicodedata.normalize("NFKC", s or "").casefold()
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", s).strip()

# ---------------- 1. FILTRO ----------------
def score_item(it):
    title, body = norm(it["title"]), norm(it["text"])
    cats, score = {}, 0
    for cid, c in TAX["categories"].items():
        hits = 0
        for k in c["keywords"]:
            if re.search(r"\b" + re.escape(k) + r"\b", title): hits += 2
            elif re.search(r"\b" + re.escape(k) + r"\b", body): hits += 1
        if hits: cats[cid] = hits; score += hits
    return score, cats

def filter_items(items, max_items=250, min_score=2, per_source=60):
    keep, out = [], []
    for it in items:
        s, cats = score_item(it)
        if s >= min_score: keep.append({**it, "prefilter_score": s, "prefilter_categories": cats})
        else: out.append({"id": it["id"], "url": it["url"], "source": it["source"], "reason": f"score {s} < {min_score}"})
    keep.sort(key=lambda x: -x["prefilter_score"])
    final, cnt = [], {}
    for it in keep:
        if cnt.get(it["source"], 0) >= per_source or len(final) >= max_items:
            out.append({"id": it["id"], "url": it["url"], "source": it["source"], "reason": "tope por fuente/total"}); continue
        cnt[it["source"]] = cnt.get(it["source"], 0) + 1; final.append(it)
    return final, out

def cmd_filter(a):
    items = read_jsonl(RAWF)
    final, out = filter_items(items, a.max)
    write_jsonl(PROC / "candidates.jsonl", final); write_jsonl(PROC / "excluded.jsonl", out)
    by = {}
    for i in final: by[i["source"]] = by.get(i["source"], 0) + 1
    print(f"Crudos: {len(items)} | candidatos: {len(final)} | excluidos: {len(out)}"); print("Por fuente:", by)

# ---------------- 2. EXTRACCION (Gemini) ----------------
API = "https://generativelanguage.googleapis.com/v1beta/"
def rank_models(models):
    """Ordena modelos flash: primero los estables, luego lite, al final preview/omni (que suelen NO estar en el plan gratuito)."""
    bad = re.compile(r"image|tts|live|audio|embed|thinking|exp|robotics|computer")
    out = []
    for m in models:
        name = m["name"].split("/")[-1]
        if "flash" not in name or "generateContent" not in m.get("supportedGenerationMethods", []) or bad.search(name): continue
        rank = 2 if re.search(r"preview|omni", name) else (1 if "lite" in name else 0)
        out.append((rank, name))
    out.sort(key=lambda x: (x[0], [-ord(c) for c in x[1]]))   # rank asc, nombre desc (versiones nuevas primero)
    return [n for _, n in out]

def list_models(key):
    req = urllib.request.Request(API + "models?pageSize=200", headers={"x-goog-api-key": key})
    return json.loads(urllib.request.urlopen(req, timeout=30).read())["models"]

def http_msg(e):
    try: return json.loads(e.read())["error"]["message"][:220]
    except Exception: return str(e)

def pick_model(key, tries=6):
    """Prueba los modelos candidatos con un pedido minimo y devuelve el primero que responde."""
    cands = rank_models(list_models(key))
    if not cands: raise RuntimeError("No encontre modelos flash disponibles; fijalo con --model")
    for name in cands[:tries]:
        try:
            gemini_call('Responde solo este JSON: {"ok": true}', name, key)
            print(f"  [OK]    {name}"); return name
        except urllib.error.HTTPError as e: print(f"  [FALLA] {name}: HTTP {e.code} - {http_msg(e)}")
        except Exception as e: print(f"  [FALLA] {name}: {type(e).__name__}: {e}")
        time.sleep(2)
    raise RuntimeError("Ningun modelo respondio. Revisa la clave/cuota en aistudio.google.com (Rate limits) o fija uno con --model")

def gemini_call(prompt, model, key):
    body = {"contents": [{"parts": [{"text": prompt}]}], "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}}
    req = urllib.request.Request(API + f"models/{model}:generateContent", data=json.dumps(body).encode(),
                                 headers={"x-goog-api-key": key, "Content-Type": "application/json"})
    r = json.loads(urllib.request.urlopen(req, timeout=90).read())
    return r["candidates"][0]["content"]["parts"][0]["text"]

def build_prompt(it):
    cats = "; ".join(f'{k} ({v["label"]})' for k, v in TAX["categories"].items())
    return f"""Sos un analista que ayuda a decidir que tecnologias incorporar en un laboratorio de investigacion en salud/biotecnologia (tipo CONICET).
REGLAS ESTRICTAS:
- Usa SOLO el texto de abajo. No uses conocimiento propio ni inventes datos.
- Si el texto no trata una tecnologia o tendencia relevante para un laboratorio de investigacion, devolve {{"trends": []}}.
- Maximo 2 tendencias. Cada una DEBE incluir "quote": una cita copiada EXACTAMENTE del texto (8 a 40 palabras, sin modificar).
- companies, countries y amounts: solo si aparecen literalmente en el texto; si no, listas vacias.
- horizon y hype_phase son tu JUICIO: justificalo en una frase ("justification") apoyada en el texto.
Valores permitidos: category = {cats}; horizon = corto (0-2 anios) | mediano (2-5) | largo (5-10+); hype_phase = {" | ".join(TAX["hype_phases"])}.
Responde SOLO JSON con esta forma: {{"trends":[{{"technology":"","category":"","horizon":"","hype_phase":"","justification":"","quote":"","companies":[],"countries":[],"amounts":[]}}]}}

TITULO: {it['title']}
FUENTE: {it['source']} ({it['source_type']})
TEXTO: {it['text']}"""

def cmd_extract(a):
    key = os.environ.get("GEMINI_API_KEY")
    if not key: sys.exit("Falta GEMINI_API_KEY (en Colab: Secretos -> GEMINI_API_KEY).")
    print("Probando modelos disponibles para tu clave...")
    model = pick_model(key) if a.model == "auto" else a.model
    print("Modelo elegido:", model)
    cands = read_jsonl(PROC / "candidates.jsonl"); done = {r["item_id"] for r in read_jsonl(PROC / "extractions.jsonl")}
    todo = [c for c in cands if c["id"] not in done][: a.limit]
    print(f"Pendientes: {len([c for c in cands if c['id'] not in done])} | esta corrida: {len(todo)}")
    for n, it in enumerate(todo, 1):
        res, err = None, None
        for attempt in range(4):
            try:
                res = json.loads(gemini_call(build_prompt(it), model, key)); break
            except urllib.error.HTTPError as e:
                if e.code == 429:
                    if attempt == 3: print("Cuota agotada (" + http_msg(e) + "). Volve a correr mas tarde: retoma donde quedo."); return
                    time.sleep(30 * (attempt + 1)); continue
                err = f"HTTP {e.code}: {http_msg(e)}"; break
            except Exception as e: err = f"{type(e).__name__}: {e}"; break
        write_jsonl(PROC / "extractions.jsonl", [{"item_id": it["id"], "model": model, "extracted_at": NOW(), "result": res, "error": err}], "a")
        if n % 10 == 0: print(f"  {n}/{len(todo)}")
        time.sleep(a.sleep)

# ---------------- 3. VERIFICACION ----------------
def nums(s): return re.findall(r"\d[\d.,]*\d|\d", s or "")

def verify_trend(t, it):
    """Devuelve (trend_limpio o None, lista de problemas)."""
    log, blob = [], norm(it["title"] + " " + it["text"])
    def P(problem, action): log.append({"item_id": it["id"], "url": it["url"], "technology": t.get("technology", ""), "problem": problem, "action": action})
    q = norm(t.get("quote", ""))
    if len(q.split()) < 6: P("cita ausente o muy corta", "descartada"); return None, log
    if q not in blob: P("la cita NO aparece en el texto crudo (posible alucinacion)", "descartada"); return None, log
    if t.get("category") not in TAX["categories"]: P(f"categoria invalida: {t.get('category')}", "descartada"); return None, log
    if t.get("horizon") not in TAX["horizons"]: P(f"horizonte invalido: {t.get('horizon')}", "descartada"); return None, log
    if t.get("hype_phase") not in TAX["hype_phases"]: P(f"fase invalida: {t.get('hype_phase')}", "descartada"); return None, log
    if not str(t.get("technology", "")).strip(): P("sin nombre de tecnologia", "descartada"); return None, log
    c = dict(t)
    for field in ("companies", "countries"):
        keep = []
        for v in t.get(field, []) or []:
            if norm(v) in blob: keep.append(v)
            else: P(f"{field}: '{v}' no figura en el texto", "quitado")
        c[field] = keep
    am = []
    for v in t.get("amounts", []) or []:
        if all(n in (it["title"] + " " + it["text"]) for n in nums(v)) and nums(v): am.append(v)
        else: P(f"monto '{v}' no figura en el texto", "quitado")
    c["amounts"] = am
    flags = []
    if str(it.get("published", ""))[:4].isdigit() and int(str(it["published"])[:4]) > datetime.now().year:
        flags.append("fecha_futura"); P(f"fecha de publicacion futura ({it['published']})", "marcado")
    c.update({"item_id": it["id"], "url": it["url"], "source": it["source"], "published": it.get("published", ""),
              "title": it["title"], "evidence_level": TAX["evidence_level"].get(it["source_type"], "s/d"),
              "region": it.get("region", "global"), "flags": flags, "status": "parcial" if any(l["action"] == "quitado" for l in log) else "verificado"})
    return c, log

def cmd_verify(a):
    raw = {i["id"]: i for i in read_jsonl(RAWF)}
    ok, logs, n_tr, n_err = [], [], 0, 0
    for e in read_jsonl(PROC / "extractions.jsonl"):
        if e.get("error") or not e.get("result"): n_err += 1; continue
        for t in e["result"].get("trends", []) or []:
            n_tr += 1
            c, lg = verify_trend(t, raw[e["item_id"]]) if e["item_id"] in raw else (None, [{"item_id": e["item_id"], "problem": "item no esta en el crudo", "action": "descartada"}])
            logs += lg
            if c: ok.append(c)
    write_jsonl(PROC / "verified_trends.jsonl", ok); write_jsonl(PROC / "verification_log.jsonl", logs)
    desc = sum(1 for l in logs if l["action"] == "descartada")
    print(f"Tendencias propuestas por Gemini: {n_tr} | verificadas: {len(ok)} | descartadas: {desc} | items con error: {n_err}")
    print(f"Problemas registrados: {len(logs)}  -> {PROC/'verification_log.jsonl'}")

# ---------------- autoprueba ----------------
def selftest():
    items = [{"id": "a", "url": "https://x.org/a", "title": "CRISPR base editing advances", "text": "x", "source": "pubmed", "source_type": "paper"},
             {"id": "b", "url": "https://x.org/b", "title": "Hospital parking rules", "text": "nothing relevant here", "source": "who_news", "source_type": "regulacion"}]
    f, o = filter_items(items); assert [i["id"] for i in f] == ["a"] and o[0]["id"] == "b", "filtro"
    it = {"id": "i1", "url": "https://x.org/1", "title": "Acme raises funds", "source": "stat", "source_type": "noticia", "published": "2027-01-01",
          "text": "Acme Bio raised $50 million to scale its autonomous laboratory platform in Argentina and Chile this year."}
    good = {"technology": "Laboratorio autonomo", "category": "automatizacion_lab", "horizon": "corto", "hype_phase": "Innovation Trigger", "justification": "j",
            "quote": "Acme Bio raised $50 million to scale its autonomous laboratory platform", "companies": ["Acme Bio", "FakeCorp"], "countries": ["Argentina"], "amounts": ["$50 million", "$900 million"]}
    c, lg = verify_trend(good, it)
    assert c and c["companies"] == ["Acme Bio"] and c["amounts"] == ["$50 million"] and c["status"] == "parcial" and "fecha_futura" in c["flags"], "limpieza"
    bad = dict(good, quote="Acme Bio raised $90 million to dominate the entire laboratory market worldwide")
    c2, lg2 = verify_trend(bad, it); assert c2 is None and "alucin" in lg2[0]["problem"], "cita falsa"
    c3, _ = verify_trend(dict(good, horizon="ayer"), it); assert c3 is None, "enum"
    ms = [{"name": "models/" + n, "supportedGenerationMethods": ["generateContent"]} for n in
          ["gemini-omni-flash-preview", "gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-3-flash", "gemini-2.5-flash-image", "gemini-2.5-pro", "gemini-live-flash"]]
    r = rank_models(ms); assert r[0] == "gemini-3-flash" and r[-1] == "gemini-omni-flash-preview" and "gemini-2.5-flash-image" not in r and "gemini-2.5-pro" not in r, r
    print("SELFTEST OK (ranking de modelos sin preview/omni primero, filtro, cita falsa descartada, entidades/montos inventados quitados, fecha futura marcada)")

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("cmd", choices=["filter", "extract", "verify", "selftest"])
    ap.add_argument("--max", type=int, default=250); ap.add_argument("--limit", type=int, default=40)
    ap.add_argument("--model", default="auto"); ap.add_argument("--sleep", type=float, default=4.5)
    a = ap.parse_args(); {"filter": cmd_filter, "extract": cmd_extract, "verify": cmd_verify, "selftest": lambda _: selftest()}[a.cmd](a)

## 1. Autopruebas
Deben decir `SELFTEST OK` las dos.

In [ ]:
!python /content/radar/agent/collect.py --selftest
!python /content/radar/agent/process.py selftest

## 2. Etapa 1 — Recolectar
Tabla OK/FALLA por fuente. Se puede reejecutar: no duplica.

In [ ]:
!python /content/radar/agent/collect.py --days 30

## 3. Etapa 2a — Filtrar lo relevante
Cada exclusión queda con su motivo en `excluded.jsonl`.

In [ ]:
!python /content/radar/agent/process.py filter --max 250

## 4. Etapa 2b — Extracción con Gemini
Carga la clave desde Secretos. Prueba los modelos disponibles y usa el primero que responde. `--limit` = cuántos ítems en esta corrida; si se agota la cuota diaria, corta y la próxima corrida retoma donde quedó.

In [ ]:
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('Clave cargada:', bool(os.environ['GEMINI_API_KEY']))

In [ ]:
!python /content/radar/agent/process.py extract --limit 40

## 5. Etapa 2c — Verificación
Contrasta cada tendencia con el texto crudo. Lo que no se respalda se descarta o se quita, y queda en `verification_log.jsonl`.

In [ ]:
!python /content/radar/agent/process.py verify

## 6. Ver resultados

In [ ]:
import json, pandas as pd
from pathlib import Path
P = Path('/content/radar/data/processed')
rd = lambda f: [json.loads(l) for l in (P/f).read_text(encoding='utf-8').splitlines() if l.strip()] if (P/f).exists() else []
v, lg = pd.DataFrame(rd('verified_trends.jsonl')), pd.DataFrame(rd('verification_log.jsonl'))
print('Tendencias verificadas:', len(v))
if len(v):
    display(v.groupby(['category','horizon']).size().unstack(fill_value=0))
    display(v[['technology','category','horizon','hype_phase','status','evidence_level','source','url']].head(40))
print('\nProblemas detectados por el verificador:', len(lg))
if len(lg):
    display(lg.groupby(['problem','action']).size().sort_values(ascending=False).head(15))

## 7. Guardar copia en tu Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Radar_Salud
!cp -r /content/radar/data/raw /content/radar/data/processed /content/drive/MyDrive/Radar_Salud/
print('Copiado a Drive > Radar_Salud')